# 08 — Build FAISS index from fine-tuned CLIP-L/14

Notebook này dùng checkpoint tốt nhất từ notebook `07b_finetune_clip_l14_last_layers_with_plots.ipynb` để build FAISS index cho bài toán text-to-image retrieval.

Mục tiêu:

```text
Fine-tuned CLIP-L/14 checkpoint
→ Encode toàn bộ ảnh Flickr30K
→ L2-normalize image embeddings
→ Build FAISS IndexFlatIP
→ Search ảnh bằng text query
→ Optional: evaluate Recall@1 / Recall@5 / Recall@10 bằng FAISS
```

Lưu ý:

- Vì embeddings đã được L2-normalize, `IndexFlatIP` tương đương cosine similarity.
- FAISS giúp search nhanh hơn, không làm embedding thông minh hơn.
- Chất lượng retrieval đến từ fine-tuned CLIP-L/14; FAISS là tầng truy xuất nhanh.

Bản này đã sửa để đọc đúng metadata dạng `image_path`, giống notebook 07b.


## 1. Imports

In [1]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor

try:
    import faiss
except ImportError as exc:
    raise ImportError(
        "FAISS chưa được cài. Hãy cài bằng một trong các lệnh sau:\n"
        "pip install faiss-cpu\n"
        "hoặc nếu môi trường hỗ trợ GPU: pip install faiss-gpu"
    ) from exc

d:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Config

Notebook mặc định dùng checkpoint:

```text
data/processed/clip_l14_finetune/clip_l14_last_layers_1/best_model
```

Nếu bạn lưu checkpoint ở nơi khác, chỉ cần sửa `BEST_MODEL_DIR`.
Notebook đã tự dò thư mục ảnh trong các path phổ biến như `data/raw/Images` hoặc `data/raw/flickr30k_images`.


In [2]:
# Notebook thường nằm trong folder notebooks/, nên project root là "..".
# Nếu bạn chạy notebook ngay tại project root, đoạn fallback bên dưới sẽ tự xử lý.
PROJECT_ROOT = Path("..").resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path(".").resolve()

METADATA_PATH = PROJECT_ROOT / "data/processed/metadata.json"
# Flickr30K image folder có thể có nhiều tên khác nhau tùy nguồn tải dữ liệu.
# Project hiện tại của bạn đang dùng: data/raw/Images
CANDIDATE_IMAGE_DIRS = [
    PROJECT_ROOT / "data/raw/flickr30k_images",
    PROJECT_ROOT / "data/raw/Images",
    PROJECT_ROOT / "data/raw/images",
    PROJECT_ROOT / "data/raw/Flickr30k_images",
]

IMAGE_DIR = None
for candidate_dir in CANDIDATE_IMAGE_DIRS:
    if candidate_dir.exists():
        IMAGE_DIR = candidate_dir
        break

if IMAGE_DIR is None:
    IMAGE_DIR = CANDIDATE_IMAGE_DIRS[0]

# Checkpoint tốt nhất từ notebook 07b.
BEST_MODEL_DIR = PROJECT_ROOT / "data/processed/clip_l14_finetune/clip_l14_last_layers_1/best_model"

OUTPUT_DIR = PROJECT_ROOT / "data/processed/faiss_finetuned_clip_l14"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EMBEDDINGS_PATH = OUTPUT_DIR / "image_embeddings_finetuned_clip_l14.npy"
INDEX_PATH = OUTPUT_DIR / "faiss_index_finetuned_clip_l14.index"
INDEX_METADATA_PATH = OUTPUT_DIR / "faiss_index_metadata.json"
SEARCH_RESULTS_PATH = OUTPUT_DIR / "sample_search_results.csv"
EVAL_RESULTS_PATH = OUTPUT_DIR / "faiss_recall_results.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# None = dùng toàn bộ Flickr30K.
# Đặt 1000 nếu muốn smoke test nhanh.
MAX_IMAGES = None

IMAGE_BATCH_SIZE = 16
TEXT_BATCH_SIZE = 64

# Optional FAISS evaluation.
RUN_FAISS_EVAL = True
EVAL_QUERY_BATCH_SIZE = 512
K_VALUES = (1, 5, 10)

print(f"Project root: {PROJECT_ROOT}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Image dir: {IMAGE_DIR}")
print(f"Best model dir: {BEST_MODEL_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device: {DEVICE}")

Project root: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search
Metadata path: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\metadata.json
Image dir: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\raw\Images
Best model dir: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_l14_finetune\clip_l14_last_layers_1\best_model
Output dir: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\faiss_finetuned_clip_l14
Device: cuda


## 3. Load metadata

In [3]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Cannot find metadata file: {METADATA_PATH}")

if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        "Cannot find image directory. Tried:\n"
        + "\n".join(str(p) for p in CANDIDATE_IMAGE_DIRS)
    )

if not BEST_MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Cannot find fine-tuned checkpoint: {BEST_MODEL_DIR}\n"
        "Hãy chạy notebook 07b trước, hoặc sửa BEST_MODEL_DIR cho đúng checkpoint."
    )

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

if MAX_IMAGES is not None:
    metadata = metadata[:MAX_IMAGES]

print(f"Number of images: {len(metadata):,}")
print(f"Number of captions: {sum(len(item['captions']) for item in metadata):,}")
print("Metadata keys:", list(metadata[0].keys()))
print("Example item:")
metadata[0]

Number of images: 31,782
Number of captions: 158,910
Metadata keys: ['image_id', 'image_path', 'captions']
Example item:


{'image_id': '1000092795.jpg',
 'image_path': 'data/raw/Images/1000092795.jpg',
 'captions': ['Two young guys with shaggy hair look at their hands while hanging out in the yard .',
  'Two young , White males are outside near many bushes .',
  'Two men in green shirts are standing in a yard .',
  'A man in a blue shirt standing in a garden .',
  'Two friends enjoy time spent together .']}

## 4. Load fine-tuned CLIP-L/14 checkpoint

In [4]:
model = CLIPModel.from_pretrained(BEST_MODEL_DIR).to(DEVICE)
processor = CLIPProcessor.from_pretrained(BEST_MODEL_DIR)
model.eval()

print("Loaded fine-tuned model.")
print(f"Model device: {next(model.parameters()).device}")

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 20956.78it/s]


Loaded fine-tuned model.
Model device: cuda:0


## 5. Helper functions

`encode_images()` tạo image embeddings từ fine-tuned model.

Điểm quan trọng:

```text
image → vision_model → visual_projection → L2 normalize
```

Sau khi normalize, inner product tương đương cosine similarity.

In [5]:
def get_image_path_from_item(item, project_root, image_dir=None):
    """
    Metadata của project hiện tại dùng key: image_path.
    Một số phiên bản khác có thể dùng key: image / filename.
    Hàm này giúp notebook chạy được với nhiều format metadata hơn.
    """
    if "image_path" in item:
        return Path(project_root) / item["image_path"]

    if "image" in item:
        if image_dir is None:
            return Path(project_root) / item["image"]
        return Path(image_dir) / item["image"]

    if "filename" in item:
        if image_dir is None:
            return Path(project_root) / item["filename"]
        return Path(image_dir) / item["filename"]

    raise KeyError(
        "Cannot find image path key in metadata item. "
        f"Available keys: {list(item.keys())}. "
        "Expected one of: image_path, image, filename."
    )


def get_image_name_from_item(item):
    if "image_path" in item:
        return Path(item["image_path"]).name
    if "image" in item:
        return item["image"]
    if "filename" in item:
        return item["filename"]
    return "unknown"


def encode_images(model, processor, metadata_items, project_root, image_dir, batch_size, device):
    model.eval()
    all_embeddings = []

    for start in tqdm(range(0, len(metadata_items), batch_size), desc="Encoding images"):
        batch_items = metadata_items[start:start + batch_size]
        images = []

        for item in batch_items:
            image_path = get_image_path_from_item(
                item=item,
                project_root=project_root,
                image_dir=image_dir,
            )
            with Image.open(image_path) as image:
                images.append(image.convert("RGB"))

        inputs = processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        with torch.no_grad():
            vision_outputs = model.vision_model(pixel_values=pixel_values)
            pooled_output = vision_outputs.pooler_output
            image_features = model.visual_projection(pooled_output)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        all_embeddings.append(image_features.cpu().numpy().astype(np.float32))

    return np.vstack(all_embeddings).astype(np.float32)


def encode_texts(model, processor, texts, batch_size, device):
    model.eval()
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Encoding texts"):
        batch_texts = texts[start:start + batch_size]

        inputs = processor(
            text=batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            pooled_output = text_outputs.pooler_output
            text_features = model.text_projection(pooled_output)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        all_embeddings.append(text_features.cpu().numpy().astype(np.float32))

    return np.vstack(all_embeddings).astype(np.float32)


def build_captions_and_gt_indices(metadata_items):
    captions = []
    gt_indices = []

    for image_index, item in enumerate(metadata_items):
        for caption in item["captions"]:
            captions.append(caption)
            gt_indices.append(image_index)

    return captions, np.array(gt_indices, dtype=np.int64)

## 6. Encode image embeddings

In [6]:
if IMAGE_EMBEDDINGS_PATH.exists():
    print(f"Loading cached image embeddings from: {IMAGE_EMBEDDINGS_PATH}")
    image_embeddings = np.load(IMAGE_EMBEDDINGS_PATH)
else:
    start_time = time.time()
    image_embeddings = encode_images(
        model=model,
        processor=processor,
        metadata_items=metadata,
        project_root=PROJECT_ROOT,
        image_dir=IMAGE_DIR,
        batch_size=IMAGE_BATCH_SIZE,
        device=DEVICE,
    )
    elapsed = time.time() - start_time

    np.save(IMAGE_EMBEDDINGS_PATH, image_embeddings)
    print(f"Saved image embeddings to: {IMAGE_EMBEDDINGS_PATH}")
    print(f"Image encoding time: {elapsed:.2f} seconds")

print("Image embeddings shape:", image_embeddings.shape)
print("Embedding dtype:", image_embeddings.dtype)

# Kiểm tra nhanh norm gần 1.
norms = np.linalg.norm(image_embeddings[:10], axis=1)
print("First 10 embedding norms:", norms)

Encoding images: 100%|██████████| 1987/1987 [12:12<00:00,  2.71it/s]


Saved image embeddings to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\faiss_finetuned_clip_l14\image_embeddings_finetuned_clip_l14.npy
Image encoding time: 732.96 seconds
Image embeddings shape: (31782, 768)
Embedding dtype: float32
First 10 embedding norms: [0.99999994 1.         1.         1.         1.         0.99999994
 1.         1.         1.         1.        ]


## 7. Build FAISS index

Ở đây dùng `IndexFlatIP`:

```text
Flat = exact search
IP = inner product
```

Vì embeddings đã normalize, inner product chính là cosine similarity.

In [7]:
embedding_dim = image_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(image_embeddings)

faiss.write_index(index, str(INDEX_PATH))

index_metadata = {
    "model_checkpoint": str(BEST_MODEL_DIR),
    "num_images": len(metadata),
    "embedding_dim": int(embedding_dim),
    "index_type": "IndexFlatIP",
    "image_embeddings_path": str(IMAGE_EMBEDDINGS_PATH),
    "index_path": str(INDEX_PATH),
    "note": "Image embeddings are L2-normalized, so inner product is equivalent to cosine similarity.",
}

with INDEX_METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(index_metadata, f, indent=2)

print(index)
print(f"Saved FAISS index to: {INDEX_PATH}")
print(f"Saved index metadata to: {INDEX_METADATA_PATH}")

<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000002190A8248A0> >
Saved FAISS index to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\faiss_finetuned_clip_l14\faiss_index_finetuned_clip_l14.index
Saved index metadata to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\faiss_finetuned_clip_l14\faiss_index_metadata.json


## 8. Text-to-image search demo

In [8]:
def search_images(query, model, processor, index, metadata_items, top_k=5, device=DEVICE):
    text_embedding = encode_texts(
        model=model,
        processor=processor,
        texts=[query],
        batch_size=1,
        device=device,
    )

    scores, indices = index.search(text_embedding, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        item = metadata_items[int(idx)]
        results.append({
            "rank": rank,
            "score": float(score),
            "image_index": int(idx),
            "image": get_image_name_from_item(item),
            "captions": item["captions"],
        })

    return results


sample_queries = [
    "A dog is running through the grass.",
    "A group of people are sitting at a table.",
    "A child is playing outside.",
]

all_sample_rows = []

for query in sample_queries:
    print("=" * 100)
    print("Query:", query)

    results = search_images(
        query=query,
        model=model,
        processor=processor,
        index=index,
        metadata_items=metadata,
        top_k=5,
        device=DEVICE,
    )

    for result in results:
        print(f"Rank {result['rank']} | score={result['score']:.4f} | image={result['image']}")
        print("Caption example:", result["captions"][0])

        all_sample_rows.append({
            "query": query,
            "rank": result["rank"],
            "score": result["score"],
            "image_index": result["image_index"],
            "image": result["image"],
            "caption_example": result["captions"][0],
        })

sample_df = pd.DataFrame(all_sample_rows)
sample_df.to_csv(SEARCH_RESULTS_PATH, index=False)
sample_df

Query: A dog is running through the grass.


Encoding texts: 100%|██████████| 1/1 [00:00<00:00,  7.81it/s]


Rank 1 | score=0.2865 | image=2844747252.jpg
Caption example: Two dogs are fighting over a toy and another dog is chasing them .
Rank 2 | score=0.2832 | image=2982928615.jpg
Caption example: The dog is running quickly through the meadow .
Rank 3 | score=0.2819 | image=2534502836.jpg
Caption example: A medium sized brown and white streaked dog is running through tall grass .
Rank 4 | score=0.2810 | image=2460159430.jpg
Caption example: A dog running in long grass , a housing development behind it .
Rank 5 | score=0.2793 | image=2557507575.jpg
Caption example: A brown dog running in field of long green grass .
Query: A group of people are sitting at a table.


Encoding texts: 100%|██████████| 1/1 [00:00<00:00, 78.22it/s]


Rank 1 | score=0.3104 | image=239051548.jpg
Caption example: People sit in a restaurant with tall ceilings and large windows drinking various wines and socializing .
Rank 2 | score=0.3057 | image=686293997.jpg
Caption example: A group of people sitting around a rectangular table having either pieces of paper or laptops in front of them .
Rank 3 | score=0.2881 | image=434171515.jpg
Caption example: A group of young men and women chat and observe the menu at a fancy restaurant .
Rank 4 | score=0.2872 | image=2084851847.jpg
Caption example: A group of middle-aged women and men sit around a table with beverages and documents having a meeting .
Rank 5 | score=0.2870 | image=58121544.jpg
Caption example: A group of men and women sit at a restaurant table with beer and food .
Query: A child is playing outside.


Encoding texts: 100%|██████████| 1/1 [00:00<00:00, 78.14it/s]

Rank 1 | score=0.2606 | image=3350002347.jpg
Caption example: A young boy is quite excited in the throes of a ballgame .
Rank 2 | score=0.2602 | image=3508413697.jpg
Caption example: Young boy wearing an Elmo t-shirt kneeling at a table with his meal .
Rank 3 | score=0.2569 | image=2774875318.jpg
Caption example: A boy swings from a rope suspended from a tree in a park .
Rank 4 | score=0.2529 | image=2481367956.jpg
Caption example: Small boy reaching for blue bar in play area , another child in background .
Rank 5 | score=0.2509 | image=4769586761.jpg
Caption example: A little boy is playing with a steering wheel attached to a climbing apparatus in the park .


,query,rank,score,image_index,image,caption_example
0,A dog is running through the grass.,1,0.286460,9109,2844747252.jpg,Two dogs are fighting over a toy and another d...
1,A dog is running through the grass.,2,0.283196,10217,2982928615.jpg,The dog is running quickly through the meadow .
2,A dog is running through the grass.,3,0.281886,6645,2534502836.jpg,A medium sized brown and white streaked dog is...
3,A dog is running through the grass.,4,0.280972,6077,2460159430.jpg,"A dog running in long grass , a housing develo..."
4,A dog is running through the grass.,5,0.279252,6898,2557507575.jpg,A brown dog running in field of long green gra...
5,A group of people are sitting at a table.,1,0.310390,5456,239051548.jpg,People sit in a restaurant with tall ceilings ...
6,A group of people are sitting at a table.,2,0.305661,29241,686293997.jpg,A group of people sitting around a rectangular...
7,A group of people are sitting at a table.,3,0.288051,19324,434171515.jpg,A group of young men and women chat and observ...
8,A group of people are sitting at a table.,4,0.287171,3290,2084851847.jpg,A group of middle-aged women and men sit aroun...
9,A group of people are sitting at a table.,5,0.286978,27670,58121544.jpg,A group of men and women sit at a restaurant t...


## 9. Optional: evaluate Recall@K with FAISS

Cell này đánh giá chất lượng retrieval bằng chính FAISS index.

Với `IndexFlatIP`, đây là exact search nên Recall@K nên tương đương với cách tính similarity brute-force, nhưng chạy theo interface FAISS.

In [9]:
def evaluate_faiss_recall_at_k(
    model,
    processor,
    index,
    metadata_items,
    text_batch_size,
    query_batch_size,
    device,
    k_values=(1, 5, 10),
):
    captions, gt_indices = build_captions_and_gt_indices(metadata_items)

    recall_counts = {k: 0 for k in k_values}
    num_queries = len(captions)
    max_k = max(k_values)

    start_time = time.time()

    for start in tqdm(range(0, num_queries, query_batch_size), desc="FAISS Recall@K"):
        end = min(start + query_batch_size, num_queries)
        batch_texts = captions[start:end]
        gt_batch = gt_indices[start:end]

        text_embeddings = encode_texts(
            model=model,
            processor=processor,
            texts=batch_texts,
            batch_size=text_batch_size,
            device=device,
        )

        scores, indices = index.search(text_embeddings, max_k)

        for k in k_values:
            topk = indices[:, :k]
            correct = np.any(topk == gt_batch[:, None], axis=1)
            recall_counts[k] += int(correct.sum())

    elapsed = time.time() - start_time

    results = {
        f"Recall@{k}": recall_counts[k] / num_queries
        for k in k_values
    }
    results.update({
        "num_images": len(metadata_items),
        "num_queries": num_queries,
        "elapsed_seconds": elapsed,
        "index_type": "IndexFlatIP",
        "model_checkpoint": str(BEST_MODEL_DIR),
    })

    return results


if RUN_FAISS_EVAL:
    faiss_eval_results = evaluate_faiss_recall_at_k(
        model=model,
        processor=processor,
        index=index,
        metadata_items=metadata,
        text_batch_size=TEXT_BATCH_SIZE,
        query_batch_size=EVAL_QUERY_BATCH_SIZE,
        device=DEVICE,
        k_values=K_VALUES,
    )

    with EVAL_RESULTS_PATH.open("w", encoding="utf-8") as f:
        json.dump(faiss_eval_results, f, indent=2)

    display(faiss_eval_results)
    print(f"Saved FAISS evaluation results to: {EVAL_RESULTS_PATH}")
else:
    print("RUN_FAISS_EVAL is False. Skipping FAISS evaluation.")

FAISS Recall@K: 100%|██████████| 311/311 [02:38<00:00,  1.96it/s]


{'Recall@1': 0.36524447800641874,
 'Recall@5': 0.6034044427663457,
 'Recall@10': 0.6970864011075452,
 'num_images': 31782,
 'num_queries': 158910,
 'elapsed_seconds': 158.71804523468018,
 'index_type': 'IndexFlatIP',
 'model_checkpoint': 'D:\\CITD\\HK3\\Python for ML\\Clip_Retrieval\\clip-faiss-search\\data\\processed\\clip_l14_finetune\\clip_l14_last_layers_1\\best_model'}

Saved FAISS evaluation results to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\faiss_finetuned_clip_l14\faiss_recall_results.json


## 10. Final artifacts

Sau khi chạy notebook này, các file chính được tạo trong:

```text
data/processed/faiss_finetuned_clip_l14/
```

Bao gồm:

```text
image_embeddings_finetuned_clip_l14.npy
faiss_index_finetuned_clip_l14.index
faiss_index_metadata.json
sample_search_results.csv
faiss_recall_results.json
```

Cách dùng trong app/demo:

```text
1. Load fine-tuned CLIP-L/14 checkpoint
2. Load FAISS index
3. Encode query text bằng model fine-tuned
4. Search top-k ảnh bằng FAISS
5. Trả về ảnh + caption + similarity score
```